# 15 · Sentence Window Retriever

Match a sentence precisely, then expand to the local window.

**Analogy handbook:** [sentence-window](../retriever-analogy-handbook.html#sentence-window)  
**Prerequisite:** run `00_basics_concepts.ipynb` once (or the setup cells below) so the Chroma index exists.

### Learning loop
1. Skim the analogy for this technique  
2. Run setup (reuse index if possible)  
3. Run the practical cells  
4. Ask: *Did this fix the failure mode, or only reshuffle noise?*


## Shared setup

These cells install packages, load the Llama 2 PDF, build/load the Chroma index, and define helpers.

> Prefer `REBUILD_INDEX = False` after the first successful build so later method notebooks reuse the same store.


### Learning: !pip install langchain_community langchain_text_splitters langchain_op

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Installs required Python packages into the runtime.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.2 MB/s eta 0:00:00


### Learning: IMPORTS

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `IMPORTS` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
print("All imports and setup starting...")

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import getpass
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains.hyde.base import (
    HypotheticalDocumentEmbedder
)

All imports and setup starting...


/tmp/ipykernel_520/2286258816.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: from google.colab import userdata

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `from google.colab import userdata` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: OPENAI API KEY

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `OPENAI API KEY` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 2. OPENAI API KEY
# ============================================================

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: DATA DIRECTORY

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `DATA DIRECTORY` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 3. DATA DIRECTORY
# ============================================================

DATA_DIR = Path(
    r"/content/"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

### Learning: FIND PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `FIND PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 4. FIND PDF
# ============================================================

if preferred_pdf.exists():

    PDF_PATH = preferred_pdf

else:

    available_pdfs = sorted(
        DATA_DIR.glob("*.pdf")
    )

    if len(available_pdfs) == 1:

        PDF_PATH = available_pdfs[0]

    elif len(available_pdfs) == 0:

        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )

    else:

        raise RuntimeError(
            "Multiple PDF files were found. "
            "Please set PDF_PATH manually.\n"
            + "\n".join(
                str(path)
                for path in available_pdfs
            )
        )


print("PDF found:")
print(PDF_PATH)

PDF found:
/content/llama2-research-paper.pdf


### Learning: LOAD PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `LOAD PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
# ============================================================
# 5. LOAD PDF
# ============================================================

loader = PyPDFLoader(
    str(PDF_PATH)
)

pages = loader.load()

print(
    f"\nTotal PDF pages loaded: {len(pages)}"
)


# ============================================================
# 6. INSPECT FIRST PAGE
# ============================================================

print("\nFirst-page metadata:")
print(
    pages[0].metadata
)

print("\nFirst 1,000 characters:")
print(
    pages[0].page_content[:1000]
)


Total PDF pages loaded: 77

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Art

### Learning: IDENTIFY PAPER SECTIONS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: IDENTIFY PAPER SECTIONS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. IDENTIFY PAPER SECTIONS
# ============================================================

def identify_section(
    paper_page: int
) -> str:

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


# ============================================================
# 8. ADD METADATA
# ============================================================

for page_document in pages:

    page_index = int(
        page_document.metadata.get(
            "page",
            0
        )
    )

    paper_page = (
        page_index + 1
    )

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(
                paper_page
            ),
            "access_level": "public",
        }
    )


print("\nMetadata after enrichment:")

for page_document in pages[:5]:

    print(
        page_document.metadata
    )



Metadata after enrichment:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',

### Learning: TEXT SPLITTING

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEXT SPLITTING` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(
    pages
)

print(
    f"\nTotal pages: {len(pages)}"
)

print(
    f"Total chunks: {len(chunks)}"
)


# ============================================================
# 10. ADD CHUNK IDs
# ============================================================

for chunk_number, chunk in enumerate(
    chunks
):

    paper_page = chunk.metadata.get(
        "paper_page",
        "unknown"
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"llama2-page-"
        f"{paper_page}-"
        f"chunk-{chunk_number}"
    )


print("\nFirst chunk content:")

print(
    chunks[0].page_content[:1000]
)

print("\nFirst chunk metadata:")

print(
    chunks[0].metadata
)


Total pages: 77
Total chunks: 343

First chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Shar

### Learning: CREATE EMBEDDING MODEL

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE EMBEDDING MODEL` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 11. CREATE EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 12. TEST EMBEDDING MODEL
# ============================================================

test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(
    f"\nEmbedding dimensions: "
    f"{len(test_vector)}"
)

print(
    f"First 10 values: "
    f"{test_vector[:10]}"
)



Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


### Learning: CHROMA CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CHROMA CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 13. CHROMA CONFIGURATION
# ============================================================

PERSIST_DIRECTORY = (
    DATA_DIR
    / "chroma_llama2_retriever"
)

COLLECTION_NAME = (
    "llama2_retriever_demo"
)


### Learning: CREATE OR LOAD VECTOR STORE

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE OR LOAD VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 14. CREATE OR LOAD VECTOR STORE
# ============================================================

# True  = rebuild complete vector DB
# False = reuse existing vector DB

REBUILD_INDEX = True

### Learning: VERIFY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `VERIFY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
if REBUILD_INDEX:

    print(
        "\nRebuilding vector store..."
    )

    if PERSIST_DIRECTORY.exists():

        shutil.rmtree(
            PERSIST_DIRECTORY,
            ignore_errors=True
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(
            PERSIST_DIRECTORY
        ),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

    print(
        "New vector store created."
    )

else:

    if not PERSIST_DIRECTORY.exists():

        print(
            "\nExisting vector DB "
            "not found."
        )

        print(
            "Creating a new vector store..."
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
            collection_configuration={
                "hnsw": {
                    "space": "cosine"
                }
            },
        )

        print(
            "New vector store created."
        )

    else:

        print(
            "\nLoading existing "
            "vector store..."
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
        )

        print(
            "Existing vector store "
            "loaded."
        )


# ============================================================
# 15. VERIFY VECTOR STORE
# ============================================================

stored_count = (
    vector_store
    ._collection
    .count()
)

print(
    f"\nStored chunks: "
    f"{stored_count}"
)

print(
    f"Persisted at: "
    f"{PERSIST_DIRECTORY}"
)



Rebuilding vector store...
New vector store created.

Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-37-08-Aug-2026-prompting\data\chroma_llama2_retriever


### Learning: sentence window retriever practical

**What you'll learn:** Match a sentence, return the surrounding window for context.

**What this cell does:** Runs `sentence window retriever practical` and prints intermediate results you can inspect.

**Watch for:** Precision of the match + context of the neighborhood.



In [ ]:
# sentence window retriever practical

### Learning: SENTENCE WINDOW RETRIEVAL PRACTICAL

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `SENTENCE WINDOW RETRIEVAL PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
# ============================================================
# SENTENCE WINDOW RETRIEVAL PRACTICAL
# ============================================================

# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

import re
import uuid

from langchain_core.documents import Document
from langchain_chroma import Chroma


### Learning: CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

# Number of sentences before and after the matched sentence
WINDOW_SIZE = 2

SENTENCE_WINDOW_COLLECTION = "sentence_window_retriever_demo"

SENTENCE_WINDOW_PERSIST_DIR = (
    DATA_DIR / "sentence_window_chroma"
)


### Learning: SENTENCE SPLITTING FUNCTION

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Defines helper logic for: SENTENCE SPLITTING FUNCTION.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 3. SENTENCE SPLITTING FUNCTION
# ============================================================

def split_into_sentences(text: str) -> list[str]:
    """
    Simple sentence splitter.

    Splits text after:
    .
    !
    ?

    while keeping reasonably clean sentence boundaries.
    """

    text = text.strip()

    if not text:
        return []

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


### Learning: CREATE SENTENCE-WINDOW DOCUMENTS

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `CREATE SENTENCE-WINDOW DOCUMENTS` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
# ============================================================
# 4. CREATE SENTENCE-WINDOW DOCUMENTS
# ============================================================

sentence_documents = []


for page_document in pages:

    sentences = split_into_sentences(
        page_document.page_content
    )

    for sentence_index, sentence in enumerate(sentences):

        # --------------------------------------------
        # Calculate surrounding sentence range
        # --------------------------------------------

        start_index = max(
            0,
            sentence_index - WINDOW_SIZE
        )

        end_index = min(
            len(sentences),
            sentence_index + WINDOW_SIZE + 1
        )

        # --------------------------------------------
        # Build surrounding context window
        # --------------------------------------------

        window_sentences = sentences[
            start_index:end_index
        ]

        sentence_window = " ".join(
            window_sentences
        )

        # --------------------------------------------
        # Copy original metadata
        # --------------------------------------------

        metadata = dict(
            page_document.metadata
        )

        metadata.update(
            {
                "sentence_index": sentence_index,
                "window_start": start_index,
                "window_end": end_index - 1,
                "sentence_window": sentence_window,
                "original_sentence": sentence,
                "window_size": WINDOW_SIZE,
            }
        )

        # --------------------------------------------
        # IMPORTANT:
        # page_content contains ONLY the sentence.
        #
        # This small sentence is what gets embedded
        # and searched.
        # --------------------------------------------

        sentence_document = Document(
            page_content=sentence,
            metadata=metadata
        )

        sentence_documents.append(
            sentence_document
        )


print(
    f"Total sentence documents created: "
    f"{len(sentence_documents)}"
)


Total sentence documents created: 2910


### Learning: INSPECT ONE SENTENCE DOCUMENT

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `INSPECT ONE SENTENCE DOCUMENT` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:


# ============================================================
# 5. INSPECT ONE SENTENCE DOCUMENT
# ============================================================

example_document = sentence_documents[20]

print("\nSEARCHABLE SENTENCE:")
print(
    example_document.page_content
)

print("\nSURROUNDING WINDOW:")
print(
    example_document.metadata[
        "sentence_window"
    ]
)

print("\nMETADATA:")
print(
    example_document.metadata
)



SEARCHABLE SENTENCE:
.

SURROUNDING WINDOW:
. . . . .

METADATA:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 1, 'page_label': '2', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 2, 'section': 'front_matter', 'access_level': 'public', 'sentence_index': 14, 'window_start': 12, 'window_end': 16, 'sentence_window': '. . . . .', 'original_sentence': '.', 'window_size': 2}


### Learning: CREATE SENTENCE-LEVEL VECTOR STORE

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Runs `CREATE SENTENCE-LEVEL VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ============================================================
# 6. CREATE SENTENCE-LEVEL VECTOR STORE
# ============================================================

sentence_vector_store = Chroma.from_documents(
    documents=sentence_documents,
    embedding=embeddings,
    collection_name=SENTENCE_WINDOW_COLLECTION,
    persist_directory=str(
        SENTENCE_WINDOW_PERSIST_DIR
    ),
)


print(
    "\nSentence-level vector store created."
)


Sentence-level vector store created.


### Learning: CREATE BASE SENTENCE RETRIEVER

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `CREATE BASE SENTENCE RETRIEVER` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 7. CREATE BASE SENTENCE RETRIEVER
# ============================================================

sentence_retriever = (
    sentence_vector_store
    .as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 4
        }
    )
)


print(
    "Sentence retriever created successfully."
)


Sentence retriever created successfully.


### Learning: USER QUERY

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `USER QUERY` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
# ============================================================
# 8. USER QUERY
# ============================================================

query = (
    "How was Llama 2 aligned using human feedback?"
)

print("\nUSER QUERY:")
print(query)


USER QUERY:
How was Llama 2 aligned using human feedback?


### Learning: RETRIEVE MATCHING SENTENCES

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: RETRIEVE MATCHING SENTENCES.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. RETRIEVE MATCHING SENTENCES
# ============================================================

matched_sentences = (
    sentence_retriever.invoke(
        query
    )
)


print(
    "\nMATCHED SENTENCES"
)

print(
    "=" * 100
)


for i, document in enumerate(
    matched_sentences,
    start=1
):

    print(
        f"\nMATCH {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Sentence index:",
        document.metadata.get(
            "sentence_index"
        )
    )

    print(
        "\nMatched sentence:"
    )

    print(
        document.page_content
    )


MATCHED SENTENCES

MATCH 1
Page: 8
Sentence index: 19

Matched sentence:
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources.

MATCH 2
Page: 10
Sentence index: 26

Matched sentence:
Leveraging such response scores as rewards, we can optimizeLlama 2-Chat during RLHF for
better human preference alignment and improved helpfulness and safety.

MATCH 3
Page: 4
Sentence index: 6

Matched sentence:
Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data.

MATCH 4
Page: 3
Sentence index: 0

Matched sentence:
Figure 1: Helpfulness human evaluationresults forLlama
2-Chatcompared to other open-source and closed-source
models.


### Learning: REPLACE MATCHED SENTENCE WITH SENTENCE WINDOW

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `REPLACE MATCHED SENTENCE WITH SENTENCE WINDOW` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 10. REPLACE MATCHED SENTENCE WITH SENTENCE WINDOW
# ============================================================

window_documents = []


for matched_document in matched_sentences:

    window_text = (
        matched_document
        .metadata
        .get(
            "sentence_window",
            matched_document.page_content
        )
    )

    window_document = Document(
        page_content=window_text,
        metadata={
            **matched_document.metadata,

            "matched_sentence":
                matched_document.page_content,

            "retrieval_type":
                "sentence_window"
        }
    )

    window_documents.append(
        window_document
    )


### Learning: DISPLAY SENTENCE WINDOW RESULTS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `DISPLAY SENTENCE WINDOW RESULTS` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 11. DISPLAY SENTENCE WINDOW RESULTS
# ============================================================

print(
    "\n\nSENTENCE WINDOW RESULTS"
)

print(
    "=" * 100
)


for i, document in enumerate(
    window_documents,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Paper page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Sentence index:",
        document.metadata.get(
            "sentence_index"
        )
    )

    print(
        "\nMatched sentence:"
    )

    print(
        document.metadata.get(
            "matched_sentence"
        )
    )

    print(
        "\nReturned sentence window:"
    )

    print(
        document.page_content
    )

    print(
        "\n" + "-" * 100
    )



SENTENCE WINDOW RESULTS

RESULT 1
Paper page: 8
Sentence index: 19

Matched sentence:
3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources.

Returned sentence window:
Results for the
PaLM-2-L are from Anil et al. (2023). 3 Fine-tuning
Llama 2-Chat is the result of several months of research and iterative applications of alignment techniques,
including both instruction tuning and RLHF, requiring significant computational and annotation resources. In this section, we report on our experiments and findings using supervised fine-tuning (Section 3.1), as
well as initial and iterative reward modeling (Section 3.2.2) and RLHF (Section 3.2.3). We also share a
new technique, Ghost Attention (GAtt), which we find helps control dialogue flow over multiple turns
(Section 3.3).

----------------------------------------------

### Learning: CREATE A REUSABLE SENTENCE WINDOW RETRIEVER FUNCTION

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Defines helper logic for: CREATE A REUSABLE SENTENCE WINDOW RETRIEVER FUNCTION.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ============================================================
# 12. CREATE A REUSABLE SENTENCE WINDOW RETRIEVER FUNCTION
# ============================================================

def sentence_window_retrieve(
    query: str,
    k: int = 4
):
    """
    Search using individual sentence embeddings,
    but return each matched sentence together with
    its surrounding context window.
    """

    # --------------------------------------------
    # Retrieve the best matching sentences
    # --------------------------------------------

    retriever = (
        sentence_vector_store
        .as_retriever(
            search_type="similarity",
            search_kwargs={
                "k": k
            }
        )
    )

    matched_documents = (
        retriever.invoke(
            query
        )
    )

    # --------------------------------------------
    # Expand each sentence into its context window
    # --------------------------------------------

    expanded_documents = []

    for document in matched_documents:

        window_text = (
            document.metadata.get(
                "sentence_window",
                document.page_content
            )
        )

        expanded_document = Document(
            page_content=window_text,
            metadata={
                **document.metadata,

                "matched_sentence":
                    document.page_content,

                "retrieval_type":
                    "sentence_window"
            }
        )

        expanded_documents.append(
            expanded_document
        )

    return expanded_documents


### Learning: TEST THE REUSABLE FUNCTION

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEST THE REUSABLE FUNCTION` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 13. TEST THE REUSABLE FUNCTION
# ============================================================

query = (
    "How does reinforcement learning improve Llama 2-Chat?"
)

results = sentence_window_retrieve(
    query=query,
    k=4
)


print(
    "\nREUSABLE SENTENCE WINDOW RETRIEVER"
)

print(
    "=" * 100
)


for i, document in enumerate(
    results,
    start=1
):

    print(
        f"\nRESULT {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "\nMatched sentence:"
    )

    print(
        document.metadata.get(
            "matched_sentence"
        )
    )

    print(
        "\nFull sentence window:"
    )

    print(
        document.page_content
    )

    print(
        "\n" + "-" * 100
    )



REUSABLE SENTENCE WINDOW RETRIEVER

RESULT 1
Page: 13

Matched sentence:
Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat.

Full sentence window:
We note that reward model accuracy is one of the most
important proxies for the final performance ofLlama 2-Chat. While best practices for comprehensively
evaluating a generative model is an open research question, the ranking task of the reward has no ambiguity. Therefore, everything else being equal, an improvement of the reward model can be directly translated into
an improvement forLlama 2-Chat. 3.2.3 Iterative Fine-Tuning
As we received more batches of human preference data annotation, we were able to train better reward
models and collect more prompts. We therefore trained successive versions for RLHF models, referred to
here as RLHF-V1, ..., RLHF-V5.

-------------------------------------------------------------------------------------------------

### Learning: COMPARE NORMAL SENTENCE SEARCH VS SENTENCE WINDOW

**What you'll learn:** Match a sentence, return the surrounding window for context.

**What this cell does:** Executes retrieval/generation for: COMPARE NORMAL SENTENCE SEARCH VS SENTENCE WINDOW.

**Watch for:** Precision of the match + context of the neighborhood.



In [ ]:
# ============================================================
# 14. COMPARE NORMAL SENTENCE SEARCH VS SENTENCE WINDOW
# ============================================================

query = (
    "How was human preference data "
    "used to improve Llama 2?"
)


# ------------------------------------------------------------
# Normal sentence retrieval
# ------------------------------------------------------------

normal_sentence_results = (
    sentence_retriever.invoke(
        query
    )
)


# ------------------------------------------------------------
# Sentence window retrieval
# ------------------------------------------------------------

sentence_window_results = (
    sentence_window_retrieve(
        query=query,
        k=4
    )
)


### Learning: DISPLAY COMPARISON

**What you'll learn:** Match a sentence, return the surrounding window for context.

**What this cell does:** Runs `DISPLAY COMPARISON` and prints intermediate results you can inspect.

**Watch for:** Precision of the match + context of the neighborhood.



In [ ]:
# ============================================================
# 15. DISPLAY COMPARISON
# ============================================================

print(
    "\nNORMAL SENTENCE RETRIEVAL"
)

print(
    "=" * 100
)


for i, document in enumerate(
    normal_sentence_results,
    start=1
):

    print(
        f"\nResult {i}:"
    )

    print(
        document.page_content
    )


print(
    "\n\nSENTENCE WINDOW RETRIEVAL"
)

print(
    "=" * 100
)


for i, document in enumerate(
    sentence_window_results,
    start=1
):

    print(
        f"\nResult {i}:"
    )

    print(
        document.page_content
    )



NORMAL SENTENCE RETRIEVAL

Result 1:
As we collected more preference data, our
reward models improved, and we were able to train progressively better versions forLlama 2-Chat (see
the results in Section 5, Figure 20).Llama 2-Chat improvement also shifted the model’s data distribution.

Result 2:
Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data.

Result 3:
Leveraging such response scores as rewards, we can optimizeLlama 2-Chat during RLHF for
better human preference alignment and improved helpfulness and safety.

Result 4:
This reflects the nature of our
iterative model update and preference data annotation procedure - with better-performingLlama 2-Chat
models used for response sampling over time, it becomes challenging for annotators to select a better one
from two equally high-quality responses.


SENTENCE WINDOW RETRIEVAL

Result 1:
Safety guidelines and more detailed information regarding safety annotations
can be found in Section 4.2.1. Human 

### Learning: FINAL CONCEPTUAL FLOW

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `FINAL CONCEPTUAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 16. FINAL CONCEPTUAL FLOW
# ============================================================

"""
SENTENCE WINDOW RETRIEVAL

Document
    ↓
Split into individual sentences
    ↓
For every sentence:
    store surrounding sentences in metadata
    ↓
Embed ONLY individual sentences
    ↓
User Query
    ↓
Vector Search
    ↓
Best matching sentence
    ↓
Read sentence_window from metadata
    ↓
Return:
previous sentences
+
matched sentence
+
next sentences
    ↓
LLM
"""


print(
    "\nSentence Window Retrieval "
    "practical completed successfully."
)

### Learning: Sentence 8 → Llama 2 first undergoes supervised fine-tuning.

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `Sentence 8 → Llama 2 first undergoes supervised fine-tuning.` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
Sentence 8 → Llama 2 first undergoes supervised fine-tuning.
Sentence 9 → Human preference data is collected.
Sentence 10 → RLHF is used to align Llama 2-Chat.
Sentence 11 → Reward models score candidate responses.
Sentence 12 → PPO is used for optimization.

### Learning: Sentence 10

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `Sentence 10` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
Sentence 10
→ "RLHF is used to align Llama 2-Chat."

### Learning: {

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `{` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
{
    "sentence_window":
    """
    Llama 2 first undergoes supervised fine-tuning.
    Human preference data is collected.
    RLHF is used to align Llama 2-Chat.
    Reward models score candidate responses.
    PPO is used for optimization.
    """
}

### Learning: How is Llama 2 aligned using human feedback?

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `How is Llama 2 aligned using human feedback?` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
How is Llama 2 aligned using human feedback?

### Learning: Sentence 10

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `Sentence 10` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
Sentence 10

### Learning: Sentence 8

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Runs `Sentence 8` and prints intermediate results you can inspect.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
Sentence 8
Sentence 9
Sentence 10
Sentence 11
Sentence 12